# Chapters 4–7 bilingual knowledge graph (Kaggle)

This copy keeps the original notebook’s classification and graph cells. The added preparation stage reads the supplied English/Hindi extracted chapter text, asks Groq to identify aligned concepts with short evidence excerpts, and saves a restartable audited-JSON-style dataset before the original classifier runs. Set Kaggle secret `groq_api_key1` (additional numbered keys are optional). For a Kaggle background run, use **Run All** and keep Internet enabled.

Input files are found recursively under `/kaggle/input`; outputs and caches go under `/kaggle/working/ncert_prerequisite_kg`. The run is limited to chapters 4–7.


In [9]:
import json
from pathlib import Path
from IPython.display import FileLink, display

root = Path('/kaggle/working/ncert_prerequisite_kg')
data = json.loads(
    (root / 'bilingual_chapters_4_7_audited.json').read_text(encoding='utf-8')
)

for chapter in ('CH05', 'CH06'):
    concepts = [
        c for c in data['concepts']
        if c.get('chapter_id') == chapter
    ]
    passage_ids = {
        str(pid)
        for c in concepts
        for field in ('english_passage_ids', 'hindi_passage_ids')
        for pid in c.get(field, [])
    }
    passages = [
        p for p in data['passages']
        if str(p.get('passage_id')) in passage_ids
    ]

    out = root / f'bilingual_{chapter}_audited.json'
    out.write_text(
        json.dumps(
            {'concepts': concepts, 'passages': passages},
            ensure_ascii=False,
            indent=2
        ),
        encoding='utf-8'
    )
    print(chapter, 'concepts:', len(concepts))
    display(FileLink(out.relative_to('/kaggle/working').as_posix()))

CH05 concepts: 73


/kaggle/working/ncert_prerequisite_kg/bilingual_CH05_audited.json

CH06 concepts: 75


/kaggle/working/ncert_prerequisite_kg/bilingual_CH06_audited.json

In [10]:
from pathlib import Path
import shutil
import zipfile

root = Path('/kaggle/working/ncert_prerequisite_kg')
root.mkdir(parents=True, exist_ok=True)

# Copy cache files if Kaggle has unpacked them.
for saved in Path('/kaggle/input').rglob('concepts_CH*.json'):
    target = root / saved.name
    if not target.exists():
        shutil.copy2(saved, target)

# Restore cache files from the attached ZIP, including filenames ending in "(1)".
for archive in Path('/kaggle/input').rglob('*.zip'):
    if 'chapter_4_7_concept_checkpoint' not in archive.name:
        continue
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            name = Path(member).name
            if name.startswith('concepts_CH') and name.endswith('.json'):
                target = root / name
                if not target.exists():
                    target.write_bytes(z.read(member))

# Rebuild CH07's final file from its saved batches.
# Keep the batch files so completed work is reused.
final_ch07 = root / 'concepts_CH07.json'
if final_ch07.exists():
    final_ch07.unlink()

print('CH04 cache ready:', (root / 'concepts_CH04.json').exists())
print('CH07 saved batches:', len(list(root.glob('concepts_CH07_batch*.json'))))

CH04 cache ready: True
CH07 saved batches: 10


In [12]:
# Prepare bilingual concept/evidence data for chapters 4–7.
# Sends small page batches to Groq and caches each batch for background-run resume.

import os, re, json, time, subprocess, sys
from pathlib import Path

try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "requests"])
    import requests

ROOT = Path('/kaggle/working/ncert_prerequisite_kg') if Path('/kaggle').exists() else Path.cwd()/'ncert_prerequisite_kg'
ROOT.mkdir(parents=True, exist_ok=True)


import shutil
import zipfile

# Restore checkpoint files from the attached Kaggle input, if needed.
restored = 0

# Handle files Kaggle may have already unpacked.
for saved_file in Path('/kaggle/input').rglob('concepts_CH*.json'):
    destination = ROOT / saved_file.name
    if not destination.exists():
        shutil.copy2(saved_file, destination)
        restored += 1

# Handle the checkpoint ZIP if it remains zipped in the input.
for archive in Path('/kaggle/input').rglob('chapter_4_7_concept_checkpoint.zip'):
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            filename = Path(name).name
            if filename.startswith('concepts_CH') and filename.endswith('.json'):
                destination = ROOT / filename
                if not destination.exists():
                    destination.write_bytes(z.read(name))
                    restored += 1

print('Restored checkpoint files:', restored)
print('Available concept cache files:', len(list(ROOT.glob('concepts_CH*.json'))))



CHAPTERS = range(4, 8)
MODEL = 'openai/gpt-oss-120b'
API_URL = 'https://api.groq.com/openai/v1/chat/completions'

def find_text(lang, chapter):
    code = f'{100 + chapter:03d}'
    roots = [Path('/kaggle/input'), Path.cwd()]
    matches = []
    for root in roots:
        if root.exists():
            matches += list(root.rglob(f'{lang}_*{code}_clean.txt'))
    if not matches:
        raise FileNotFoundError(
            f'Missing {lang} text for chapter {chapter}; expected code {code}.'
        )
    return sorted(matches, key=lambda p: len(str(p)))[0].read_text(encoding='utf-8')

def get_api_key():
    key = os.environ.get('GROQ_API_KEY') or os.environ.get('groq_api_key1')
    try:
        from kaggle_secrets import UserSecretsClient
        key = key or UserSecretsClient().get_secret('groq_api_key1')
    except Exception:
        pass
    if not key:
        raise RuntimeError(
            'Add Kaggle secret groq_api_key1 and enable notebook access.'
        )
    return key.strip()

def split_pages(text):
    # Extract the text between markers such as "===== PAGE 1 =====".
    matches = list(re.finditer(r'===== PAGE\s+(\d+)\s+=====', text))
    pages = {}
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        pages[int(match.group(1))] = text[start:end].strip()
    if not pages:
        # Fallback if markers are missing.
        pages = {1: text[:5000]}
    return pages

def ask_groq(key, chapter, page_numbers, en_text, hi_text):
    prompt = {
        'task': (
            'Identify distinct mathematics concepts taught in this textbook excerpt. '
            'Align English and Hindi names only when the same concept is supported '
            'by both excerpts. Return JSON only with key "concepts", an array of '
            'objects containing name_en, name_hi, evidence_en, evidence_hi. '
            'Evidence must be short verbatim text from the matching language. '
            'If a language excerpt does not support a concept, leave its name or '
            'evidence empty. Exclude exercises and examples as standalone concepts.'
        ),
        'chapter_id': f'CH{chapter:02d}',
        'pages': page_numbers,
        'english_text': en_text,
        'hindi_text': hi_text
    }
    last_error = None
    for attempt in range(5):
        try:
            resp = requests.post(
                API_URL,
                headers={
                    'Authorization': 'Bearer ' + key,
                    'Content-Type': 'application/json'
                },
                json={
                    'model': MODEL,
                    'temperature': 0,
                    'response_format': {'type': 'json_object'},
                    'messages': [{
                        'role': 'user',
                        'content': json.dumps(prompt, ensure_ascii=False)
                    }]
                },
                timeout=180
            )
            if resp.status_code == 413:
                raise RuntimeError(
                    'Groq still reports payload too large; reduce PAGES_PER_BATCH to 1.'
                )
            if resp.status_code == 429 or resp.status_code >= 500:
                time.sleep(min(60, 5 * (attempt + 1)))
                last_error = f'HTTP {resp.status_code}: {resp.text[:300]}'
                continue
            resp.raise_for_status()
            content = resp.json()['choices'][0]['message']['content'].strip()
            content = re.sub(r'^```(?:json)?\s*|\s*```$', '', content, flags=re.I)
            result = json.loads(content)
            if not isinstance(result.get('concepts'), list):
                raise ValueError('Groq response has no concepts list.')
            return result['concepts']
        except requests.RequestException as exc:
            last_error = str(exc)
            time.sleep(min(30, 3 * (attempt + 1)))
    raise RuntimeError(f'Groq request failed after retries: {last_error}')

key = get_api_key()
PAGES_PER_BATCH = 2
chapters = []

for chapter in CHAPTERS:
    en_pages = split_pages(find_text('en', chapter))
    hi_pages = split_pages(find_text('hi', chapter))
    page_numbers = sorted(set(en_pages) | set(hi_pages))
    chapter_cache = ROOT / f'concepts_CH{chapter:02d}.json'

    if chapter_cache.exists():
        try:
            data = json.loads(chapter_cache.read_text(encoding='utf-8'))
            if data.get('concepts'):
                print(f'CH{chapter:02d}: using cached chapter extraction ({len(data["concepts"])} concepts)')
                chapters.append((chapter, data))
                continue
        except Exception:
            pass

    all_concepts = []
    batches = [
        page_numbers[i:i + PAGES_PER_BATCH]
        for i in range(0, len(page_numbers), PAGES_PER_BATCH)
    ]

    for batch_index, batch_pages in enumerate(batches):
        batch_cache = ROOT / (
            f'concepts_CH{chapter:02d}_batch{batch_index:03d}.json'
        )
        if batch_cache.exists():
            try:
                batch_data = json.loads(batch_cache.read_text(encoding='utf-8'))
                batch_concepts = batch_data.get('concepts', [])
                print(f'CH{chapter:02d} batch {batch_index + 1}/{len(batches)}: cached')
            except Exception:
                batch_concepts = None
        else:
            batch_concepts = None

        if batch_concepts is None:
            en_text = '\n\n'.join(
                f'PAGE {p}: {en_pages.get(p, "")}' for p in batch_pages
            )
            hi_text = '\n\n'.join(
                f'PAGE {p}: {hi_pages.get(p, "")}' for p in batch_pages
            )
            batch_concepts = ask_groq(
                key, chapter, batch_pages, en_text, hi_text
            )
            batch_cache.write_text(
                json.dumps({'concepts': batch_concepts}, ensure_ascii=False, indent=2),
                encoding='utf-8'
            )
            print(f'CH{chapter:02d} batch {batch_index + 1}/{len(batches)}: extracted {len(batch_concepts)} concepts')

        all_concepts.extend(batch_concepts)

    # Merge repeated concepts that appear on multiple pages.
    merged = {}
    for item in all_concepts:
        name = str(item.get('name_en', '')).strip()
        key_name = re.sub(r'\W+', ' ', name.lower()).strip()
        if not key_name:
            key_name = re.sub(r'\W+', ' ', str(item.get('name_hi', '')).lower()).strip()
        if not key_name:
            continue
        if key_name not in merged:
            merged[key_name] = {
                'name_en': str(item.get('name_en', '')).strip(),
                'name_hi': str(item.get('name_hi', '')).strip(),
                'evidence_en': str(item.get('evidence_en', '')).strip(),
                'evidence_hi': str(item.get('evidence_hi', '')).strip()
            }
        else:
            for field in ('evidence_en', 'evidence_hi'):
                extra = str(item.get(field, '')).strip()
                if extra and extra not in merged[key_name][field]:
                    merged[key_name][field] = (
                        merged[key_name][field] + ' ' + extra
                    ).strip()
            for field in ('name_en', 'name_hi'):
                if not merged[key_name][field] and item.get(field):
                    merged[key_name][field] = str(item[field]).strip()

    data = {'concepts': list(merged.values())}
    if not data['concepts']:
        raise ValueError(f'CH{chapter:02d}: no concepts extracted.')
    chapter_cache.write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f'CH{chapter:02d}: saved {len(data["concepts"])} merged concepts')
    chapters.append((chapter, data))

concepts = []
passages = []
for chapter, data in chapters:
    for i, item in enumerate(data['concepts'], 1):
        cid = f'CH{chapter:02d}_C{i:03d}'
        en_pid, hi_pid = f'{cid}_EN', f'{cid}_HI'
        en_ev = str(item.get('evidence_en', '')).strip()
        hi_ev = str(item.get('evidence_hi', '')).strip()
        concepts.append({
            'concept_id': cid,
            'chapter_id': f'CH{chapter:02d}',
            'name_en': str(item.get('name_en', '')).strip(),
            'name_hi': str(item.get('name_hi', '')).strip(),
            'english_passage_ids': [en_pid] if en_ev else [],
            'hindi_passage_ids': [hi_pid] if hi_ev else [],
            'semantic_validation': 'model_extracted_pending_review'
        })
        if en_ev:
            passages.append({
                'passage_id': en_pid, 'language': 'en',
                'cleaned_text': en_ev, 'pdf_page': None
            })
        if hi_ev:
            passages.append({
                'passage_id': hi_pid, 'language': 'hi',
                'cleaned_text': hi_ev, 'pdf_page': None
            })

if not concepts:
    raise RuntimeError('No concepts were prepared.')

dataset = {'concepts': concepts, 'passages': passages}
out = ROOT / 'bilingual_chapters_4_7_audited.json'
out.write_text(json.dumps(dataset, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Prepared {len(concepts)} concepts and {len(passages)} evidence passages across CH04–CH07:', out)

Restored checkpoint files: 0
Available concept cache files: 68
CH04: using cached chapter extraction (87 concepts)
CH05: using cached chapter extraction (73 concepts)
CH06: using cached chapter extraction (75 concepts)
CH07 batch 1/25: cached
CH07 batch 2/25: cached
CH07 batch 3/25: cached
CH07 batch 4/25: cached
CH07 batch 5/25: cached
CH07 batch 6/25: cached
CH07 batch 7/25: cached
CH07 batch 8/25: cached
CH07 batch 9/25: cached
CH07 batch 10/25: cached
CH07 batch 11/25: cached
CH07 batch 12/25: extracted 6 concepts
CH07 batch 13/25: extracted 6 concepts
CH07 batch 14/25: extracted 10 concepts
CH07 batch 15/25: extracted 8 concepts
CH07 batch 16/25: extracted 5 concepts
CH07 batch 17/25: extracted 6 concepts
CH07 batch 18/25: extracted 5 concepts
CH07 batch 19/25: extracted 6 concepts
CH07 batch 20/25: extracted 4 concepts
CH07 batch 21/25: extracted 6 concepts
CH07 batch 22/25: extracted 6 concepts
CH07 batch 23/25: extracted 3 concepts
CH07 batch 24/25: extracted 5 concepts
CH07 ba

In [13]:
import json
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

root = Path('/kaggle/working/ncert_prerequisite_kg')
combined = root / 'bilingual_chapters_4_7_audited.json'

if not combined.exists():
    raise FileNotFoundError(
        'Wait for the preparation cell to finish; the combined JSON is not ready yet.'
    )

data = json.loads(combined.read_text(encoding='utf-8'))
chapter_files = []

for chapter in ('CH04', 'CH05', 'CH06', 'CH07'):
    concepts = [
        c for c in data['concepts']
        if c.get('chapter_id') == chapter
    ]
    passage_ids = {
        str(pid)
        for c in concepts
        for field in ('english_passage_ids', 'hindi_passage_ids')
        for pid in c.get(field, [])
    }
    passages = [
        p for p in data['passages']
        if str(p.get('passage_id')) in passage_ids
    ]

    chapter_path = root / f'bilingual_{chapter}_audited.json'
    chapter_path.write_text(
        json.dumps(
            {'concepts': concepts, 'passages': passages},
            ensure_ascii=False,
            indent=2
        ),
        encoding='utf-8'
    )
    chapter_files.append(chapter_path)
    print(chapter, 'concepts:', len(concepts))

zip_path = root / 'chapter_4_7_concept_checkpoint.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(root.glob('concepts_CH*.json')):
        z.write(path, arcname=path.name)
    z.write(combined, arcname=combined.name)
    for path in chapter_files:
        z.write(path, arcname=path.name)

print('Checkpoint ZIP:')
display(FileLink(zip_path.relative_to('/kaggle/working').as_posix()))

print('Friend chapter files:')
for path in chapter_files:
    if path.name in ('bilingual_CH05_audited.json', 'bilingual_CH06_audited.json'):
        display(FileLink(path.relative_to('/kaggle/working').as_posix()))

CH04 concepts: 87
CH05 concepts: 73
CH06 concepts: 75
CH07 concepts: 91
Checkpoint ZIP:


/kaggle/working/ncert_prerequisite_kg/chapter_4_7_concept_checkpoint.zip

Friend chapter files:


/kaggle/working/ncert_prerequisite_kg/bilingual_CH05_audited.json

/kaggle/working/ncert_prerequisite_kg/bilingual_CH06_audited.json

In [17]:
RUN_CHAPTER = 'CH04'

In [18]:
# Broad, two-pass prerequisite search for chapters 4–7.
# Replace the old all-pairs classifier cell with this entire cell.

import os
import re
import sys
import json
import csv
import time
import html
import subprocess
from pathlib import Path

try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "requests"])
    import requests

try:
    import networkx as nx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "networkx"])
    import networkx as nx

try:
    from pyvis.network import Network
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pyvis"])
    from pyvis.network import Network

from IPython.display import display, IFrame

ROOT = Path('/kaggle/working/ncert_prerequisite_kg')
ROOT.mkdir(parents=True, exist_ok=True)

# Find the concept dataset made by the preparation cell.
dataset_matches = list(Path('/kaggle/working').rglob(
    'bilingual_chapters_4_7_audited.json'
))
if not dataset_matches:
    dataset_matches = list(Path('/kaggle/input').rglob(
        'bilingual_chapters_4_7_audited.json'
    ))
if not dataset_matches:
    raise FileNotFoundError(
        'Could not find bilingual_chapters_4_7_audited.json. '
        'Run the concept-preparation cell first.'
    )

dataset_path = sorted(dataset_matches, key=lambda p: len(str(p)))[0]
dataset = json.loads(dataset_path.read_text(encoding='utf-8'))
concepts = [
    c for c in dataset['concepts']
    if c.get('chapter_id') == RUN_CHAPTER
]
if not concepts:
    raise ValueError(f'No concepts found for {RUN_CHAPTER}.')
passage_ids = {
    str(pid)
    for c in concepts
    for field in ('english_passage_ids', 'hindi_passage_ids')
    for pid in c.get(field, [])
}
passages = [
    p for p in dataset['passages']
    if str(p.get('passage_id')) in passage_ids
]

print('Loading:', dataset_path)
print('Graph chapter:', RUN_CHAPTER)
print('Concepts retained:', len(concepts))

# One configured key is enough. Additional numbered keys are optional.
key = os.environ.get('GROQ_API_KEY') or os.environ.get('groq_api_key1')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('groq_api_key1')
except Exception:
    pass
if not key:
    raise RuntimeError(
        'Add Kaggle secret groq_api_key1 and enable notebook access.'
    )
key = key.strip()

RUN_CHAPTER = 'CH04'  # Change to 'CH07' for your next chapter-only graph.
MODEL = 'openai/gpt-oss-120b'
API_URL = 'https://api.groq.com/openai/v1/chat/completions'

passage_by_id = {str(p['passage_id']): p for p in passages}
concept_by_id = {str(c['concept_id']): c for c in concepts}
concept_ids = set(concept_by_id)

# Give the model the complete concept inventory, with short bilingual evidence.
# Compact shared inventory; detailed evidence is sent only for current targets.
inventory = []
detailed_targets = {RUN_CHAPTER: []}

for c in concepts:
    cid = str(c['concept_id'])
    inventory.append(f'{cid} | {c.get("name_en", "")}')

    target = {
        'id': cid,
        'chapter': c.get('chapter_id', ''),
        'name_en': c.get('name_en', ''),
        'name_hi': c.get('name_hi', ''),
        'evidence_en': [],
        'evidence_hi': []
    }

    for field, output_field in (
        ('english_passage_ids', 'evidence_en'),
        ('hindi_passage_ids', 'evidence_hi')
    ):
        for pid in c.get(field, [])[:1]:
            p = passage_by_id.get(str(pid))
            if p:
                text = ' '.join(str(
                    p.get('cleaned_text') or p.get('raw_text') or ''
                ).split())
                if text:
                    target[output_field].append({
                        'passage_id': str(pid),
                        'text': text[:60]
                    })

    detailed_targets[c.get('chapter_id', '')].append(target)
# Each search pass targets one chapter but sees all concepts from chapters 4–7.
passes = [
    {
        'name': 'comprehensive',
        'instruction': (
            'Search comprehensively for every plausible DIRECT prerequisite '
            'relationship involving the target chapter concepts. A source '
            'concept must be learned before the target concept. Consider '
            'concepts in the assigned chapter only. Do not use a top-k limit. '
            'Exclude mere topic similarity and transitive-only links. When a '
            'relationship is plausible but uncertain, include it with '
            'needs_review=true.'
        )
    },
    {
        'name': 'independent_audit',
        'instruction': (
            'Perform an independent high-recall audit for direct prerequisites '
            'of the target chapter concepts. Search the assigned chapter inventory from '
            'scratch and include plausible links that a first pass might miss. '
            'Do not use a top-k limit. Exclude mere topic similarity and '
            'transitive-only links. Mark uncertain candidates needs_review=true.'
        )
    }
]

def parse_model_json(content):
    content = re.sub(
        r'^```(?:json)?\s*|\s*```$',
        '',
        content.strip(),
        flags=re.I
    )
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        start = content.find('{')
        end = content.rfind('}')
        if start >= 0 and end > start:
            return json.loads(content[start:end + 1])
        raise


def search_chapter(chapter, search_pass):
    targets = detailed_targets[chapter]
    batch_size = 3
    all_edges = []

    for batch_start in range(0, len(targets), batch_size):
        batch_number = batch_start // batch_size
        target_batch = targets[batch_start:batch_start + batch_size]

        cache_path = ROOT / (
            f'broad_edges_{search_pass["name"]}_{chapter}_'
            f'batch{batch_number:02d}.json'
        )

        if cache_path.exists():
            cached = json.loads(cache_path.read_text(encoding='utf-8'))
            edges = cached.get('edges', [])
            print(
                f'{search_pass["name"]} {chapter} batch {batch_number + 1}: '
                f'using cache ({len(edges)} edges)'
            )
            all_edges.extend(edges)
            continue

        prompt = {
            'instruction': search_pass['instruction'],
            'target_concepts_with_evidence': target_batch,
            'all_possible_concepts': inventory,
            'rules': [
                'Find every plausible direct prerequisite for each target.',
                'A source concept must be understood before its target.',
                'The inventory uses the format concept_id | English name.',
                'Consider any concept in the full inventory as a possible source.',
                'Do not use a top-k limit.',
                'Exclude mere topic similarity and transitive-only links.',
                'Use only supplied concept IDs and passage IDs.',
                'Return JSON only: {"edges":[{"source_id":"...","target_id":"...",'
                '"confidence":0.0,"reasoning":"...","evidence_passage_ids":[],'
                '"needs_review":true}]}',
                'Return {"edges":[]} if no plausible edge is found.'
            ]
        }

        result = None
        for attempt in range(3):
            response = requests.post(
                API_URL,
                headers={
                    'Authorization': 'Bearer ' + key,
                    'Content-Type': 'application/json'
                },
                json={
                    'model': MODEL,
                    'temperature': 0 if attempt == 0 else 0.2,
                    'response_format': {'type': 'json_object'},
                    'max_completion_tokens': 1800,
                    'messages': [{
                        'role': 'user',
                        'content': json.dumps(prompt, ensure_ascii=False)
                    }]
                },
                timeout=300
            )
            if response.status_code == 429:
                delay = 1.0
                retry_after = response.headers.get('retry-after')
                if retry_after:
                    try:
                        delay = max(1.0, float(retry_after) + 0.5)
                    except ValueError:
                        pass
                match = re.search(
                    r'try again in ([\d.]+)\s*(ms|milliseconds|s|seconds?)',
                    response.text,
                    flags=re.I
                )
                if match and not retry_after:
                    amount = float(match.group(1))
                    unit = match.group(2).lower()
                    delay = amount / 1000 if unit.startswith('m') else amount
                    delay = max(1.0, delay + 0.5)

                print(f'Rate limited; retrying in {delay:.1f}s')
                time.sleep(delay)
                continue 

            if response.status_code == 400:
                try:
                    error = response.json().get('error', {})
                except ValueError:
                    error = {}
                if error.get('code') == 'json_validate_failed':
                    print(
                        f'Invalid JSON from model; retry '
                        f'{attempt + 1}/3 for {chapter} batch {batch_number + 1}'
                    )
                    time.sleep(5 * (attempt + 1))
                    continue

            if response.status_code != 200:
                raise RuntimeError(
                    f'Groq HTTP {response.status_code} for '
                    f'{search_pass["name"]} {chapter} batch {batch_number + 1}: '
                    f'{response.text[:1000]}'
                )

            try:
                result = parse_model_json(
                    response.json()['choices'][0]['message']['content']
                )
                break
            except (ValueError, KeyError, TypeError):
                print(
                    f'Could not parse model JSON; retry '
                    f'{attempt + 1}/3 for {chapter} batch {batch_number + 1}'
                )
                time.sleep(5 * (attempt + 1))

        if result is None:
            raise RuntimeError(
                f'Could not get valid JSON for {search_pass["name"]} '
                f'{chapter} batch {batch_number + 1} after 3 attempts.'
            )

        edges = result.get('edges')
        if not isinstance(edges, list):
            raise ValueError('Groq response has no edges list.')

        cache_path.write_text(
            json.dumps({'edges': edges}, ensure_ascii=False, indent=2),
            encoding='utf-8'
        )
        print(
            f'{search_pass["name"]} {chapter} batch {batch_number + 1}: '
            f'found {len(edges)} edges; cached'
        )
        all_edges.extend(edges)
        time.sleep(60)

    return all_edges



raw_edges = []
for search_pass in passes:
    for chapter in (RUN_CHAPTER,):
        raw_edges.extend(search_chapter(chapter, search_pass))

# Validate IDs and merge duplicate reports from the two passes.
valid_passage_ids = set(passage_by_id)
merged_edges = {}

for edge in raw_edges:
    source = str(edge.get('source_id', '')).strip()
    target = str(edge.get('target_id', '')).strip()

    if source not in concept_ids or target not in concept_ids or source == target:
        continue

    evidence_ids = edge.get('evidence_passage_ids') or []
    if not isinstance(evidence_ids, list):
        evidence_ids = []
    evidence_ids = list(dict.fromkeys(
        str(pid) for pid in evidence_ids
        if str(pid) in valid_passage_ids
    ))

    confidence = edge.get('confidence', 0)
    try:
        confidence = max(0.0, min(1.0, float(confidence)))
    except (TypeError, ValueError):
        confidence = 0.0

    edge_key = (source, target)
    reasoning = str(edge.get('reasoning', '')).strip()
    if edge_key not in merged_edges:
        merged_edges[edge_key] = {
            'source_id': source,
            'target_id': target,
            'confidence': confidence,
            'reasoning': reasoning,
            'evidence_passage_ids': evidence_ids,
            'needs_review': bool(edge.get('needs_review', True)),
            'passes_found': 1
        }
    else:
        saved = merged_edges[edge_key]
        saved['confidence'] = max(saved['confidence'], confidence)
        saved['needs_review'] = (
            saved['needs_review'] or bool(edge.get('needs_review', True))
        )
        saved['passes_found'] += 1
        saved['evidence_passage_ids'] = list(dict.fromkeys(
            saved['evidence_passage_ids'] + evidence_ids
        ))
        if reasoning and reasoning not in saved['reasoning']:
            saved['reasoning'] = (
                saved['reasoning'] + ' | ' + reasoning
            ).strip(' |')

edge_rows = []
classification_rows = []

for edge in merged_edges.values():
    src = concept_by_id[edge['source_id']]
    dst = concept_by_id[edge['target_id']]
    pair_id = f'{edge["source_id"]}__{edge["target_id"]}'

    edge_rows.append({
        'source_id': edge['source_id'],
        'target_id': edge['target_id'],
        'source_en': src.get('name_en', ''),
        'source_hi': src.get('name_hi', ''),
        'target_en': dst.get('name_en', ''),
        'target_hi': dst.get('name_hi', ''),
        'source_chapter': src.get('chapter_id', ''),
        'target_chapter': dst.get('chapter_id', ''),
        'confidence': edge['confidence'],
        'reasoning': edge['reasoning'],
        'supporting_passage_ids': ';'.join(edge['evidence_passage_ids']),
        'needs_review': edge['needs_review'],
        'validation_status': 'PENDING',
        'pair_id': pair_id,
        'passes_found': edge['passes_found'],
        'cycle_flag': False,
        'model': MODEL
    })

    classification_rows.append({
        'pair_id': pair_id,
        'concept_a_id': edge['source_id'],
        'concept_b_id': edge['target_id'],
        'concept_a_en': src.get('name_en', ''),
        'concept_b_en': dst.get('name_en', ''),
        'label': 'A_TO_B',
        'confidence': edge['confidence'],
        'reasoning': edge['reasoning'],
        'evidence_passage_ids': ';'.join(edge['evidence_passage_ids']),
        'needs_review': edge['needs_review'],
        'validation_status': 'PENDING',
        'passes_found': edge['passes_found'],
        'model': MODEL
    })

# Build graph with every concept, including concepts with no proposed edge.
G = nx.DiGraph()
for c in concepts:
    G.add_node(
        str(c['concept_id']),
        name_en=c.get('name_en', ''),
        name_hi=c.get('name_hi', ''),
        chapter=c.get('chapter_id', ''),
        semantic_validation=c.get('semantic_validation', 'pending_review')
    )

for row in edge_rows:
    G.add_edge(
        row['source_id'],
        row['target_id'],
        confidence=float(row['confidence']),
        reasoning=row['reasoning'],
        evidence_passage_ids=row['supporting_passage_ids'],
        validation_status='PENDING',
        needs_review=bool(row['needs_review']),
        passes_found=int(row['passes_found'])
    )

cycle_edges = set()
for component in nx.strongly_connected_components(G):
    if len(component) > 1:
        cycle_edges.update(G.subgraph(component).edges())

for row in edge_rows:
    row['cycle_flag'] = (
        row['source_id'], row['target_id']
    ) in cycle_edges

def save_csv(path, rows, columns):
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(
            f, fieldnames=columns, extrasaction='ignore'
        )
        writer.writeheader()
        writer.writerows(rows)

edge_columns = [
    'source_id', 'target_id', 'source_en', 'source_hi',
    'target_en', 'target_hi', 'source_chapter', 'target_chapter',
    'confidence', 'reasoning', 'supporting_passage_ids',
    'needs_review', 'validation_status', 'pair_id',
    'passes_found', 'cycle_flag', 'model'
]
save_csv(ROOT / f'prerequisite_edges_{RUN_CHAPTER}.csv', edge_rows, edge_columns)
save_csv(ROOT / 'prerequisite_edges.csv', edge_rows, edge_columns)
save_csv(
    ROOT / f'all_pair_classifications_{RUN_CHAPTER}.csv',
    classification_rows,
    [
        'pair_id', 'concept_a_id', 'concept_b_id',
        'concept_a_en', 'concept_b_en', 'label', 'confidence',
        'reasoning', 'evidence_passage_ids', 'needs_review',
        'validation_status', 'passes_found', 'model'
    ]
)
save_csv(
    ROOT / 'all_pair_classifications.csv',
    classification_rows,
    [
        'pair_id', 'concept_a_id', 'concept_b_id',
        'concept_a_en', 'concept_b_en', 'label', 'confidence',
        'reasoning', 'evidence_passage_ids', 'needs_review',
        'validation_status', 'passes_found', 'model'
    ]
)

# Save graph files for the existing visualization cells.
net = Network(
    height='800px',
    width='100%',
    directed=True,
    bgcolor='#FFFFFF',
    font_color='#172B4D',
    cdn_resources='in_line'
)
colors = {
    'CH04': '#A9D5F5',
    'CH05': '#B7E6D2',
    'CH06': '#F8D9A8',
    'CH07': '#D6C8F0'
}

for node, data in G.nodes(data=True):
    title = (
        f"<b>{html.escape(str(data.get('name_en', '')))}</b><br>"
        f"{html.escape(str(data.get('name_hi', '')))}<br>"
        f"Chapter: {html.escape(str(data.get('chapter', '')))}<br>"
        f"Concept ID: {html.escape(str(node))}"
    )
    net.add_node(
        node,
        label=str(data.get('name_en', node)),
        title=title,
        color=colors.get(data.get('chapter'), '#D5E5F1'),
        shape='dot',
        size=15
    )

for row in edge_rows:
    net.add_edge(
        row['source_id'],
        row['target_id'],
        arrows='to',
        title=html.escape(row['reasoning']),
        color='#D15A55' if row['cycle_flag'] else '#6C91B3',
        dashes=bool(row['cycle_flag']),
        width=2
    )

html_path = ROOT / f'knowledge_graph_{RUN_CHAPTER}.html'
net.save_graph(str(html_path))
nx.write_graphml(G, ROOT / f'knowledge_graph_{RUN_CHAPTER}.graphml')
# Generic aliases let the original visualization cells use the latest chapter.
net.save_graph(str(ROOT / 'knowledge_graph.html'))
nx.write_graphml(G, ROOT / 'knowledge_graph.graphml')

print('Candidate prerequisite edges:', len(edge_rows))
print('Edges found by both passes:', sum(e['passes_found'] == 2 for e in edge_rows))
print('Nodes retained:', G.number_of_nodes())
print('Cycles needing review:', len(cycle_edges))
print('Saved:', ROOT / f'prerequisite_edges_{RUN_CHAPTER}.csv')
print('Saved:', ROOT / f'knowledge_graph_{RUN_CHAPTER}.graphml')
print('Saved:', html_path)

display(IFrame(src=html_path.as_posix(), width='100%', height=800))

Loading: /kaggle/working/ncert_prerequisite_kg/bilingual_chapters_4_7_audited.json
Graph chapter: CH04
Concepts retained: 87
comprehensive CH04 batch 1: using cache (3 edges)
comprehensive CH04 batch 2: using cache (12 edges)
comprehensive CH04 batch 3: using cache (11 edges)
comprehensive CH04 batch 4: using cache (8 edges)
comprehensive CH04 batch 5: using cache (9 edges)
comprehensive CH04 batch 6: using cache (8 edges)
comprehensive CH04 batch 7: using cache (9 edges)
comprehensive CH04 batch 8: using cache (8 edges)
comprehensive CH04 batch 9: using cache (2 edges)
comprehensive CH04 batch 10: using cache (8 edges)
comprehensive CH04 batch 11: using cache (7 edges)
comprehensive CH04 batch 12: using cache (12 edges)
comprehensive CH04 batch 13: using cache (6 edges)
comprehensive CH04 batch 14: using cache (8 edges)
comprehensive CH04 batch 15: using cache (8 edges)
comprehensive CH04 batch 16: using cache (3 edges)
comprehensive CH04 batch 17: using cache (8 edges)
comprehensive 

In [19]:
from pathlib import Path
from IPython.display import HTML, display

graph_path = Path(
    '/kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04.html'
)
print('File exists:', graph_path.exists())
display(HTML(filename=str(graph_path)))

File exists: True


In [21]:
import os
from IPython.display import FileLink, display

os.chdir('/kaggle/working')
display(FileLink('ncert_prerequisite_kg/knowledge_graph_CH04.html'))

/kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04.html

In [25]:
from pathlib import Path
import networkx as nx

matches = list(Path('/kaggle/input').rglob('knowledge_graph05.graphml'))
if not matches:
    raise FileNotFoundError(
        'Could not find knowledge_graph05.graphml. '
        'Check that your friend’s output is attached as Kaggle input.'
    )

friend_graph_path = matches[0]
G = nx.read_graphml(friend_graph_path)

print('Loaded:', friend_graph_path)
print('Nodes:', G.number_of_nodes(), '| Edges:', G.number_of_edges())

Loaded: /kaggle/input/datasets/vaishnavichand13/chapter5/knowledge_graph05.graphml
Nodes: 73 | Edges: 246


In [29]:
from pathlib import Path
import networkx as nx

matches = list(Path("/kaggle/input").rglob("knowledge_graph06.graphml"))
if not matches:
    raise FileNotFoundError("Could not find knowledge_graph06.graphml under /kaggle/input")

graph_path = matches[0]
print("Loading:", graph_path)

G = nx.read_graphml(graph_path)
print(f"Loaded {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

Loading: /kaggle/input/datasets/vaishnavichand13/chapter6/knowledge_graph06.graphml
Loaded 75 nodes and 292 edges


In [31]:
from pathlib import Path

for chapter in (4, 5, 6):
    print(f"\nChapter {chapter}:")
    for path in sorted(Path("/kaggle/input").rglob("*")):
        if path.is_file() and (
            f"CH{chapter:02d}" in path.name
            or f"{chapter:02d}" in path.name
            or f"chapter{chapter}" in str(path).lower()
        ):
            print(path)


Chapter 4:
/kaggle/input/datasets/vaishnavichand13/capstone/extracted/en/en_fegp104_clean.txt
/kaggle/input/datasets/vaishnavichand13/capstone/extracted/hi/hi_fhgp104_clean.txt
/kaggle/input/datasets/vaishnavichand13/checkpts/bilingual_CH04_audited.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch000.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch001.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch002.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch003.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch004.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch005.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch006.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH04_batch007.json
/kaggle/input/datasets/vaishnavichand13/checkpts/concepts_CH0

In [ ]:
import os

output_dir = "/kaggle/working/ncert_prerequisite_kg"
print(os.listdir(output_dir))

In [22]:
import shutil
from IPython.display import FileLink, display

folder = "/kaggle/working/ncert_prerequisite_kg"
zip_path = shutil.make_archive(
    "/kaggle/working/NCERT_Knowledge_Graph",
    "zip",
    folder
)

display(FileLink(zip_path))


/kaggle/working/NCERT_Knowledge_Graph.zip

In [28]:
import shutil
import os

source = "/kaggle/working/ncert_prerequisite_kg"
destination = "/kaggle/working/NCERT_Knowledge_Graph"

zip_path = shutil.make_archive(destination, "zip", source)

print("ZIP created:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024 / 1024, 2), "MB")

ZIP created: /kaggle/working/NCERT_Knowledge_Graph.zip
Size: 0.75 MB


In [32]:
from pathlib import Path
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML, display

# Chapter 4 was created in this notebook's working folder.
ch4 = Path("/kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04.graphml")

# Chapters 5 and 6 came from your friends' Kaggle input datasets.
def find_graph(filename):
    matches = list(Path("/kaggle/input").rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    return matches[0]

graph_paths = {
    "CH04": ch4,
    "CH05": find_graph("knowledge_graph05.graphml"),
    "CH06": find_graph("knowledge_graph06.graphml"),
}

for chapter, path in graph_paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{chapter} graph not found at {path}. "
            "If the Kaggle session restarted, add your Chapter 4 graph as an input."
        )
    print(chapter, "←", path)

# Merge nodes and edges while preserving all node attributes.
G = nx.MultiDiGraph()

for chapter, path in graph_paths.items():
    chapter_graph = nx.read_graphml(path)
    print(
        chapter,
        "nodes:", chapter_graph.number_of_nodes(),
        "edges:", chapter_graph.number_of_edges()
    )

    for node, attrs in chapter_graph.nodes(data=True):
        G.add_node(node, **attrs)

    if chapter_graph.is_multigraph():
        for source, target, _, attrs in chapter_graph.edges(
            keys=True, data=True
        ):
            G.add_edge(source, target, **attrs)
    else:
        for source, target, attrs in chapter_graph.edges(data=True):
            G.add_edge(source, target, **attrs)

# Check that the merged nodes retain both language labels.
sample_node, sample_attrs = next(iter(G.nodes(data=True)))
print("Example node:", sample_node, sample_attrs)
print("Combined graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

# Create a bilingual interactive HTML view.
out = Path("/kaggle/working/ncert_prerequisite_kg")
out.mkdir(parents=True, exist_ok=True)

net = Network(height="850px", width="100%", directed=True, notebook=False)
for node, attrs in G.nodes(data=True):
    en = str(attrs.get("name_en", attrs.get("label", node)))
    hi = str(attrs.get("name_hi", ""))
    chapter = str(attrs.get("chapter_id", ""))
    net.add_node(
        str(node),
        label=en,
        title=f"{en}<br>{hi}<br>{chapter}",
        group=chapter
    )

for source, target, attrs in G.edges(data=True):
    net.add_edge(str(source), str(target), title=str(attrs))

graphml_out = out / "knowledge_graph_CH04_CH06.graphml"
html_out = out / "knowledge_graph_CH04_CH06.html"
nx.write_graphml(G, graphml_out)
net.write_html(str(html_out), notebook=False)

print("Saved:", graphml_out)
print("Saved:", html_out)
display(HTML(filename=str(html_out)))

CH04 ← /kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04.graphml
CH05 ← /kaggle/input/datasets/vaishnavichand13/chapter5/knowledge_graph05.graphml
CH06 ← /kaggle/input/datasets/vaishnavichand13/chapter6/knowledge_graph06.graphml
CH04 nodes: 87 edges: 366
CH05 nodes: 73 edges: 246
CH06 nodes: 75 edges: 292
Example node: CH04_C001 {'name_en': 'Data', 'name_hi': 'आँकड़े', 'chapter': 'CH04', 'semantic_validation': 'model_extracted_pending_review'}
Combined graph: 235 nodes, 904 edges
Saved: /kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04_CH06.graphml
Saved: /kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04_CH06.html


In [36]:
from pathlib import Path
from IPython.display import HTML, display
import html

graph_path = Path(
    "/kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04_CH06.html"
)
print("Graph file exists:", graph_path.exists())
print("File size:", graph_path.stat().st_size if graph_path.exists() else "missing")

if not graph_path.exists():
    raise FileNotFoundError("The combined graph HTML file was not created.")

graph_html = graph_path.read_text(encoding="utf-8")
iframe = f"""
<iframe
    srcdoc="{html.escape(graph_html, quote=True)}"
    width="100%"
    height="850"
    style="border:1px solid #ddd"
    sandbox="allow-scripts allow-same-origin">
</iframe>
"""
display(HTML(iframe))

Graph file exists: True
File size: 468934


In [37]:
from pathlib import Path
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML, display
import html

# Load the saved combined graph if G is not already in memory.
if "G" not in globals():
    G = nx.read_graphml(
        "/kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04_CH06.graphml"
    )

# Lay out each chapter separately, then place the three layouts side by side.
chapters = ("CH04", "CH05", "CH06")
offsets = {"CH04": -1700, "CH05": 0, "CH06": 1700}
positions = {}

for chapter in chapters:
    nodes = [n for n, a in G.nodes(data=True) if a.get("chapter") == chapter]
    if not nodes:
        continue

    subgraph = G.subgraph(nodes)
    local_positions = nx.spring_layout(
        subgraph,
        seed=42,
        scale=650,
        iterations=150
    )

    for node, (x, y) in local_positions.items():
        positions[node] = (float(x + offsets[chapter]), float(y))

# Place any nodes without a chapter in the center.
for node in G.nodes:
    positions.setdefault(node, (0.0, 0.0))

net = Network(
    height="900px",
    width="100%",
    directed=True,
    cdn_resources="in_line"
)

chapter_colors = {
    "CH04": "#B8DDF4",
    "CH05": "#BFE8D5",
    "CH06": "#F6D8AD"
}

for node, attrs in G.nodes(data=True):
    en = str(attrs.get("name_en", node))
    hi = str(attrs.get("name_hi", ""))
    chapter = str(attrs.get("chapter", ""))
    x, y = positions[node]

    # Both names appear on the node; the tooltip repeats them for readability.
    label = f"{en}\n{hi}" if hi else en
    tooltip = (
        f"<b>{html.escape(en)}</b><br>"
        f"{html.escape(hi)}<br>"
        f"Chapter: {html.escape(chapter)}<br>"
        f"ID: {html.escape(str(node))}"
    )

    net.add_node(
        str(node),
        label=label,
        title=tooltip,
        x=x,
        y=y,
        physics=False,
        color=chapter_colors.get(chapter, "#D5E5F1"),
        size=22,
        font={
            "face": "Noto Sans Devanagari, Arial, sans-serif",
            "size": 15
        }
    )

for source, target, attrs in G.edges(data=True):
    source_ch = G.nodes[source].get("chapter", "")
    target_ch = G.nodes[target].get("chapter", "")
    is_cross_chapter = source_ch != target_ch

    net.add_edge(
        str(source),
        str(target),
        arrows="to",
        color="#805AD5" if is_cross_chapter else "#9AA8B5",
        width=2 if is_cross_chapter else 1,
        title=str(attrs.get("reasoning", ""))
    )

net.set_options("""
{
  "physics": {"enabled": false},
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "dragNodes": true,
    "dragView": true,
    "zoomView": true
  },
  "edges": {"smooth": false}
}
""")

output = Path(
    "/kaggle/working/ncert_prerequisite_kg/"
    "knowledge_graph_CH04_CH06_bilingual.html"
)
net.save_graph(str(output))
print("Saved:", output)

# Embed the file in the notebook output.
graph_html = output.read_text(encoding="utf-8")
display(HTML(f"""
<iframe
    srcdoc="{html.escape(graph_html, quote=True)}"
    width="100%"
    height="900"
    style="border:1px solid #ddd"
    sandbox="allow-scripts allow-same-origin">
</iframe>
"""))

Saved: /kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04_CH06_bilingual.html


In [ ]:
# Cross-chapter prerequisite search and combined bilingual graph for Chapters 4–6.
# Completed search batches are cached and reused when this cell is rerun.

import json
import os
import re
import time
import csv
import sys
import subprocess
from pathlib import Path

try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "requests"])
    import requests

try:
    import networkx as nx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "networkx"])
    import networkx as nx

try:
    from pyvis.network import Network
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pyvis"])
    from pyvis.network import Network

from IPython.display import HTML, display

ROOT = Path("/kaggle/working/ncert_prerequisite_kg")
ROOT.mkdir(parents=True, exist_ok=True)

CHAPTERS = ("CH04", "CH05", "CH06")
MODEL = "openai/gpt-oss-120b"
API_URL = "https://api.groq.com/openai/v1/chat/completions"
BATCH_SIZE = 2

# Load the bilingual concepts and evidence for Chapters 4, 5, and 6.
concepts = []
passages = []

for chapter in CHAPTERS:
    matches = list(Path("/kaggle/input").rglob(
        f"bilingual_{chapter}_audited.json"
    ))
    if not matches:
        raise FileNotFoundError(
            f"Could not find bilingual_{chapter}_audited.json under /kaggle/input"
        )

    chapter_path = sorted(matches, key=lambda p: len(str(p)))[0]
    data = json.loads(chapter_path.read_text(encoding="utf-8"))
    concepts.extend(data.get("concepts", []))
    passages.extend(data.get("passages", []))
    print(f"{chapter}: {len(data.get('concepts', []))} concepts from {chapter_path}")

# Deduplicate concepts and passages by ID.
concepts = list({
    str(c["concept_id"]): c for c in concepts
}.values())

passage_by_id = {
    str(p["passage_id"]): p
    for p in passages
    if p.get("passage_id") is not None
}
concept_by_id = {
    str(c["concept_id"]): c for c in concepts
}
concept_ids = set(concept_by_id)

print("Total concepts:", len(concepts))

# Read the Kaggle secret.
key = os.environ.get("GROQ_API_KEY") or os.environ.get("groq_api_key1")
try:
    if not key:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("groq_api_key1")
except Exception:
    pass

if not key:
    raise RuntimeError(
        "Add Kaggle secret groq_api_key1 and enable notebook access to it."
    )
key = key.strip()

# Compact full inventory plus short evidence excerpts for each target concept.
inventory = [
    f'{c["concept_id"]} | {c.get("name_en", "")}'
    for c in concepts
]
targets_by_chapter = {chapter: [] for chapter in CHAPTERS}

for concept in concepts:
    chapter = str(concept.get("chapter_id", ""))
    if chapter not in targets_by_chapter:
        continue

    target = {
        "id": str(concept["concept_id"]),
        "chapter": chapter,
        "name_en": str(concept.get("name_en", "")),
        "name_hi": str(concept.get("name_hi", "")),
        "evidence_en": [],
        "evidence_hi": [],
    }

    for field, output_field in (
        ("english_passage_ids", "evidence_en"),
        ("hindi_passage_ids", "evidence_hi"),
    ):
        for passage_id in concept.get(field, [])[:1]:
            passage = passage_by_id.get(str(passage_id))
            if passage:
                excerpt = " ".join(
                    str(
                        passage.get("cleaned_text")
                        or passage.get("raw_text")
                        or ""
                    ).split()
                )[:45]
                if excerpt:
                    target[output_field].append(excerpt)

    targets_by_chapter[chapter].append(target)

# Strict JSON schema: GPT-OSS 120B supports this output mode.
edge_schema = {
    "type": "object",
    "properties": {
        "edges": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "source_id": {"type": "string"},
                    "target_id": {"type": "string"},
                    "confidence": {"type": "number"},
                    "reasoning": {"type": "string"},
                    "needs_review": {"type": "boolean"},
                },
                "required": [
                    "source_id",
                    "target_id",
                    "confidence",
                    "reasoning",
                    "needs_review",
                ],
                "additionalProperties": False,
            },
        }
    },
    "required": ["edges"],
    "additionalProperties": False,
}

def get_retry_delay(response):
    match = re.search(
        r"try again in ([\d.]+)\s*(ms|milliseconds|s|seconds?)",
        response.text,
        flags=re.I,
    )
    if not match:
        return 30.0
    amount = float(match.group(1))
    if match.group(2).lower().startswith("m"):
        return amount / 1000
    return amount

def parse_model_json(content):
    content = str(content or "").strip()
    content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content, flags=re.I)
    start = content.find("{")
    end = content.rfind("}")
    if start >= 0 and end > start:
        content = content[start:end + 1]
    return json.loads(content)

passes = ("comprehensive", "independent_audit")
raw_edges = []

# Search all target concepts against the full Chapter 4–6 inventory.
for pass_name in passes:
    for chapter in CHAPTERS:
        targets = targets_by_chapter[chapter]

        for start in range(0, len(targets), BATCH_SIZE):
            batch_number = start // BATCH_SIZE
            target_batch = targets[start:start + BATCH_SIZE]

            cache_path = ROOT / (
                f"cross_edges_{pass_name}_{chapter}_"
                f"batch{batch_number:03d}.json"
            )

            if cache_path.exists():
                cached = json.loads(cache_path.read_text(encoding="utf-8"))
                edges = cached.get("edges", [])
                print(
                    f"{pass_name} {chapter} batch {batch_number + 1}: "
                    f"using cache ({len(edges)} edges)"
                )
                raw_edges.extend(edges)
                continue

            prompt = {
                "task": (
                    "Find plausible DIRECT prerequisite relationships that cross "
                    "chapter boundaries. An edge source_id -> target_id means the "
                    "source concept should be learned before the target. Return "
                    "only edges whose source and target belong to different "
                    "chapters. Search broadly; do not use a top-k limit. Exclude "
                    "mere topic similarity and transitive-only links. Mark "
                    "uncertain candidates needs_review=true."
                ),
                "search_pass": pass_name,
                "target_concepts_with_evidence": target_batch,
                "full_concept_inventory": inventory,
                "rules": [
                    "Use only concept IDs from the supplied inventory.",
                    "A prerequisite edge points from source_id to target_id.",
                    "Return an object with an edges array.",
                    "Return an empty edges array if none are plausible.",
                ],
            }

            result = None
            last_error = ""

            for attempt in range(5):
                response = requests.post(
                    API_URL,
                    headers={"Authorization": "Bearer " + key},
                    json={
                        "model": MODEL,
                        "temperature": 0,
                        "reasoning_effort": "low",
                        "max_completion_tokens": 1200,
                        "response_format": {
                            "type": "json_schema",
                            "json_schema": {
                                "name": "prerequisite_edges",
                                "strict": True,
                                "schema": edge_schema,
                            },
                        },
                        "messages": [{
                            "role": "user",
                            "content": json.dumps(prompt, ensure_ascii=False),
                        }],
                    },
                    timeout=300,
                )

                if response.status_code == 429:
                    last_error = response.text[:1200]

                    if "tokens per day" in response.text.lower():
                        raise RuntimeError(
                            "Groq daily token limit reached. Completed batches "
                            "are cached; wait for the limit to reset, then rerun "
                            "this cell."
                        )

                    delay = get_retry_delay(response)
                    if delay > 120:
                        raise RuntimeError(
                            f"Groq rate limit asks for a {delay:.1f}s wait. "
                            "Completed batches are cached; rerun this cell later."
                        )

                    print(f"Rate limited; retrying in {delay + 1:.1f}s")
                    time.sleep(delay + 1)
                    continue

                if response.status_code != 200:
                    raise RuntimeError(
                        f"Groq HTTP {response.status_code}: "
                        f"{response.text[:1500]}"
                    )

                try:
                    message = response.json()["choices"][0]["message"]
                    content = message.get("content")

                    if not content:
                        last_error = (
                            "Model returned empty content: "
                            + response.text[:1000]
                        )
                        print(last_error)
                        time.sleep(3)
                        continue

                    result = parse_model_json(content)
                    break

                except (ValueError, KeyError, TypeError) as exc:
                    last_error = f"{exc}; response: {response.text[:1000]}"
                    print(
                        f"Could not parse model JSON; retry "
                        f"{attempt + 1}/5 for {chapter} batch {batch_number + 1}"
                    )
                    time.sleep(3)

            if result is None:
                raise RuntimeError(
                    f"Batch failed after retries. Last error: {last_error}"
                )

            edges = result.get("edges", [])
            if not isinstance(edges, list):
                raise ValueError("Model response did not contain an edges list.")

            cache_path.write_text(
                json.dumps({"edges": edges}, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            raw_edges.extend(edges)
            print(
                f"{pass_name} {chapter} batch {batch_number + 1}: "
                f"found {len(edges)} edges; cached"
            )
            time.sleep(5)

# Validate IDs and merge duplicate reports from both passes.
cross_edges = {}

for edge in raw_edges:
    source = str(edge.get("source_id", "")).strip()
    target = str(edge.get("target_id", "")).strip()

    if source not in concept_ids or target not in concept_ids or source == target:
        continue

    if (
        concept_by_id[source].get("chapter_id")
        == concept_by_id[target].get("chapter_id")
    ):
        continue

    try:
        confidence = max(0.0, min(1.0, float(edge.get("confidence", 0))))
    except (TypeError, ValueError):
        confidence = 0.0

    edge_key = (source, target)
    reasoning = str(edge.get("reasoning", "")).strip()
    needs_review = bool(edge.get("needs_review", True))

    if edge_key not in cross_edges:
        cross_edges[edge_key] = {
            "source_id": source,
            "target_id": target,
            "confidence": confidence,
            "reasoning": reasoning,
            "needs_review": needs_review,
            "passes_found": 1,
        }
    else:
        saved = cross_edges[edge_key]
        saved["confidence"] = max(saved["confidence"], confidence)
        saved["needs_review"] = saved["needs_review"] or needs_review
        saved["passes_found"] += 1
        if reasoning and reasoning not in saved["reasoning"]:
            saved["reasoning"] = (
                saved["reasoning"] + " | " + reasoning
            ).strip(" |")

# Create the combined graph with every bilingual concept.
G = nx.DiGraph()

for concept in concepts:
    G.add_node(
        str(concept["concept_id"]),
        name_en=str(concept.get("name_en", "")),
        name_hi=str(concept.get("name_hi", "")),
        chapter=str(concept.get("chapter_id", "")),
        semantic_validation=str(
            concept.get("semantic_validation", "pending_review")
        ),
    )

# Locate the chapter GraphMLs created by the earlier chapter runs.
graph_paths = {
    "CH04": Path(
        "/kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04.graphml"
    )
}

for chapter, filename in (
    ("CH05", "knowledge_graph05.graphml"),
    ("CH06", "knowledge_graph06.graphml"),
):
    matches = list(Path("/kaggle/input").rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input"
        )
    graph_paths[chapter] = sorted(matches, key=lambda p: len(str(p)))[0]

for chapter, graph_path in graph_paths.items():
    if not graph_path.exists():
        if chapter == "CH04":
            matches = list(
                Path("/kaggle/input").rglob("knowledge_graph_CH04.graphml")
            )
            if matches:
                graph_path = sorted(matches, key=lambda p: len(str(p)))[0]
            else:
                raise FileNotFoundError(
                    "Chapter 4 GraphML is missing. Rerun its graph-search cell "
                    "or attach its GraphML as Kaggle input."
                )

    old_graph = nx.read_graphml(graph_path)
    for source, target, attrs in old_graph.edges(data=True):
        if source in G and target in G:
            G.add_edge(source, target, **attrs)

# Add the new cross-chapter candidate edges.
for (source, target), edge in cross_edges.items():
    G.add_edge(
        source,
        target,
        confidence=edge["confidence"],
        reasoning=edge["reasoning"],
        needs_review=edge["needs_review"],
        passes_found=edge["passes_found"],
        cross_chapter=True,
    )

# Save the cross-chapter candidates as a review CSV.
csv_path = ROOT / "cross_chapter_edges_CH04_CH06.csv"
columns = [
    "source_id", "target_id", "source_en", "source_hi",
    "target_en", "target_hi", "source_chapter", "target_chapter",
    "confidence", "reasoning", "needs_review", "passes_found",
]

with csv_path.open("w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()

    for (source, target), edge in cross_edges.items():
        src = concept_by_id[source]
        dst = concept_by_id[target]
        writer.writerow({
            "source_id": source,
            "target_id": target,
            "source_en": src.get("name_en", ""),
            "source_hi": src.get("name_hi", ""),
            "target_en": dst.get("name_en", ""),
            "target_hi": dst.get("name_hi", ""),
            "source_chapter": src.get("chapter_id", ""),
            "target_chapter": dst.get("chapter_id", ""),
            "confidence": edge["confidence"],
            "reasoning": edge["reasoning"],
            "needs_review": edge["needs_review"],
            "passes_found": edge["passes_found"],
        })

# Save combined graph data.
graphml_path = ROOT / "knowledge_graph_CH04_CH06.graphml"
html_path = ROOT / "knowledge_graph_CH04_CH06_bilingual.html"
nx.write_graphml(G, graphml_path)

# Position each chapter in its own region of the graph.
chapter_offsets = {"CH04": -1700, "CH05": 0, "CH06": 1700}
positions = {}

for chapter in CHAPTERS:
    nodes = [
        node for node, attrs in G.nodes(data=True)
        if attrs.get("chapter") == chapter
    ]
    if not nodes:
        continue

    local_positions = nx.spring_layout(
        G.subgraph(nodes),
        seed=42,
        scale=650,
        iterations=150,
    )
    for node, (x, y) in local_positions.items():
        positions[node] = (float(x + chapter_offsets[chapter]), float(y))

for node in G.nodes:
    positions.setdefault(node, (0.0, 0.0))

# Render English and Hindi labels on the nodes.
net = Network(
    height="900px",
    width="100%",
    directed=True,
    cdn_resources="in_line",
)

chapter_colors = {
    "CH04": "#B8DDF4",
    "CH05": "#BFE8D5",
    "CH06": "#F6D8AD",
}

for node, attrs in G.nodes(data=True):
    en = str(attrs.get("name_en", node))
    hi = str(attrs.get("name_hi", ""))
    chapter = str(attrs.get("chapter", ""))
    x, y = positions[node]
    label = f"{en}\n{hi}" if hi else en
    title = (
        f"<b>{en}</b><br>{hi}<br>"
        f"Chapter: {chapter}<br>ID: {node}"
    )

    net.add_node(
        str(node),
        label=label,
        title=title,
        x=x,
        y=y,
        physics=False,
        color=chapter_colors.get(chapter, "#D5E5F1"),
        size=22,
        font={
            "face": "Noto Sans Devanagari, Arial, sans-serif",
            "size": 15,
        },
    )

for source, target, attrs in G.edges(data=True):
    source_chapter = G.nodes[source].get("chapter", "")
    target_chapter = G.nodes[target].get("chapter", "")
    is_cross_chapter = source_chapter != target_chapter

    net.add_edge(
        str(source),
        str(target),
        arrows="to",
        color="#805AD5" if is_cross_chapter else "#9AA8B5",
        width=2 if is_cross_chapter else 1,
        title=str(attrs.get("reasoning", "")),
    )

net.set_options("""
{
  "physics": {"enabled": false},
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "dragNodes": true,
    "dragView": true,
    "zoomView": true
  },
  "edges": {"smooth": false}
}
""")

net.save_graph(str(html_path))

print("Cross-chapter candidate edges:", len(cross_edges))
print("Combined graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print("Cross-chapter CSV:", csv_path)
print("Combined GraphML:", graphml_path)
print("Bilingual interactive graph:", html_path)

# Embed the saved HTML directly in the Kaggle output.
import html as html_module

graph_html = html_path.read_text(encoding="utf-8")
display(HTML(f"""
<iframe
    srcdoc="{html_module.escape(graph_html, quote=True)}"
    width="100%"
    height="900"
    style="border:1px solid #ddd"
    sandbox="allow-scripts allow-same-origin">
</iframe>
"""))

CH04: 87 concepts from /kaggle/input/datasets/vaishnavichand13/checkpts/bilingual_CH04_audited.json
CH05: 73 concepts from /kaggle/input/datasets/vaishnavichand13/checkpts/bilingual_CH05_audited.json
CH06: 75 concepts from /kaggle/input/datasets/vaishnavichand13/checkpts/bilingual_CH06_audited.json
Total concepts: 235
comprehensive CH04 batch 1: using cache (5 edges)
comprehensive CH04 batch 2: using cache (9 edges)
comprehensive CH04 batch 3: using cache (7 edges)
comprehensive CH04 batch 4: using cache (0 edges)
comprehensive CH04 batch 5: using cache (0 edges)
comprehensive CH04 batch 6: using cache (2 edges)
comprehensive CH04 batch 7: using cache (0 edges)
comprehensive CH04 batch 8: using cache (5 edges)
comprehensive CH04 batch 9: using cache (5 edges)
comprehensive CH04 batch 10: using cache (5 edges)
comprehensive CH04 batch 11: using cache (0 edges)
comprehensive CH04 batch 12: using cache (10 edges)
comprehensive CH04 batch 13: using cache (5 edges)
comprehensive CH04 batch 

In [33]:
from pathlib import Path
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML, display

# Chapter 4 was created in this notebook's working folder.
ch4 = Path("/kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04.graphml")

# Chapters 5 and 6 came from your friends' Kaggle input datasets.
def find_graph(filename):
    matches = list(Path("/kaggle/input").rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    return matches[0]

graph_paths = {
    "CH04": ch4,
    "CH05": find_graph("knowledge_graph05.graphml"),
    "CH06": find_graph("knowledge_graph06.graphml"),
}

for chapter, path in graph_paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{chapter} graph not found at {path}. "
            "If the Kaggle session restarted, add your Chapter 4 graph as an input."
        )
    print(chapter, "←", path)

# Merge nodes and edges while preserving all node attributes.
G = nx.MultiDiGraph()

for chapter, path in graph_paths.items():
    chapter_graph = nx.read_graphml(path)
    print(
        chapter,
        "nodes:", chapter_graph.number_of_nodes(),
        "edges:", chapter_graph.number_of_edges()
    )

    for node, attrs in chapter_graph.nodes(data=True):
        G.add_node(node, **attrs)

    if chapter_graph.is_multigraph():
        for source, target, _, attrs in chapter_graph.edges(
            keys=True, data=True
        ):
            G.add_edge(source, target, **attrs)
    else:
        for source, target, attrs in chapter_graph.edges(data=True):
            G.add_edge(source, target, **attrs)

# Check that the merged nodes retain both language labels.
sample_node, sample_attrs = next(iter(G.nodes(data=True)))
print("Example node:", sample_node, sample_attrs)
print("Combined graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

# Create a bilingual interactive HTML view.
out = Path("/kaggle/working/ncert_prerequisite_kg")
out.mkdir(parents=True, exist_ok=True)

net = Network(height="850px", width="100%", directed=True, notebook=False)
for node, attrs in G.nodes(data=True):
    en = str(attrs.get("name_en", attrs.get("label", node)))
    hi = str(attrs.get("name_hi", ""))
    chapter = str(attrs.get("chapter_id", ""))
    net.add_node(
        str(node),
        label=en,
        title=f"{en}<br>{hi}<br>{chapter}",
        group=chapter
    )

for source, target, attrs in G.edges(data=True):
    net.add_edge(str(source), str(target), title=str(attrs))

graphml_out = out / "knowledge_graph_CH04_CH06.graphml"
html_out = out / "knowledge_graph_CH04_CH06.html"
nx.write_graphml(G, graphml_out)
net.write_html(str(html_out), notebook=False)

print("Saved:", graphml_out)
print("Saved:", html_out)
display(HTML(filename=str(html_out)))

CH04 ← /kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04.graphml
CH05 ← /kaggle/input/datasets/vaishnavichand13/chapter5/knowledge_graph05.graphml
CH06 ← /kaggle/input/datasets/vaishnavichand13/chapter6/knowledge_graph06.graphml
CH04 nodes: 87 edges: 366
CH05 nodes: 73 edges: 246
CH06 nodes: 75 edges: 292
Example node: CH04_C001 {'name_en': 'Data', 'name_hi': 'आँकड़े', 'chapter': 'CH04', 'semantic_validation': 'model_extracted_pending_review'}
Combined graph: 235 nodes, 904 edges
Saved: /kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04_CH06.graphml
Saved: /kaggle/working/ncert_prerequisite_kg/knowledge_graph_CH04_CH06.html


In [26]:

import networkx as nx
from pyvis.network import Network
from IPython.display import display, IFrame
from pathlib import Path
import html
import math

# Output directory
output = Path("/kaggle/working/ncert_knowledge_graph")
output.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Calculate fixed positions using NetworkX
# --------------------------------------------------

# Reproducible layout
positions = nx.spring_layout(
    G,
    seed=42,
    k=1.8,
    iterations=500,
    scale=1000
)

# --------------------------------------------------
# 2. Detect edges involved in directed cycles
# --------------------------------------------------

cycle_edges = set()

for component in nx.strongly_connected_components(G):
    if len(component) > 1:
        for u, v in G.subgraph(component).edges():
            cycle_edges.add((u, v))

# --------------------------------------------------
# 3. Create PyVis network
# --------------------------------------------------

net = Network(
    height="850px",
    width="100%",
    directed=True,
    bgcolor="#FFFFFF",
    font_color="#263238",
    cdn_resources="in_line",
    notebook=False
)

chapter_colors = {
    "CH04": "#B8DDF4",
    "CH05": "#BFE8D5",
    "CH06": "#F6D8AD",
    "CH07": "#D6C8F0"
}

# --------------------------------------------------
# 4. Add nodes at fixed positions
# --------------------------------------------------

for node, data in G.nodes(data=True):

    name_en = str(data.get("name_en", node))
    name_hi = str(data.get("name_hi", ""))
    chapter = str(data.get("chapter", ""))

    x, y = positions[node]

    tooltip = (
        f"<b>{html.escape(name_en)}</b><br>"
        f"{html.escape(name_hi)}<br>"
        f"Chapter: {html.escape(chapter)}<br>"
        f"Concept ID: {html.escape(str(node))}"
    )

    net.add_node(
        str(node),
        label=name_en,
        title=tooltip,
        x=float(x),
        y=float(y),
        physics=False,
        color=chapter_colors.get(chapter, "#D5E5F1"),
        shape="dot",
        size=17,
        font={
            "size": 15,
            "face": "Arial"
        }
    )

# --------------------------------------------------
# 5. Add directed edges
# --------------------------------------------------

for u, v, data in G.edges(data=True):

    is_cycle = (u, v) in cycle_edges

    net.add_edge(
        str(u),
        str(v),
        arrows="to",
        color="#D9534F" if is_cycle else "#91A4B7",
        dashes=is_cycle,
        width=1.5,
        title=html.escape(
            str(data.get("reasoning", ""))
        )
    )

# --------------------------------------------------
# 6. Disable physics completely
# --------------------------------------------------

net.set_options("""
{
  "physics": {
    "enabled": false
  },
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "keyboard": true,
    "dragNodes": true,
    "dragView": true,
    "zoomView": true
  },
  "edges": {
    "smooth": false
  }
}
""")

# --------------------------------------------------
# 7. Save graph
# --------------------------------------------------

html_path = output / "knowledge_graph_stable.html"

net.save_graph(str(html_path))

print("Graph saved:", html_path)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Cycle-related edges:", len(cycle_edges))

# --------------------------------------------------
# 8. Display in Kaggle
# --------------------------------------------------

# Display the generated HTML directly
from IPython.display import HTML

display(HTML(filename=str(html_path)))


Graph saved: /kaggle/working/ncert_knowledge_graph/knowledge_graph_stable.html
Nodes: 73
Edges: 246
Cycle-related edges: 125


In [ ]:
from pathlib import Path
from IPython.display import display, HTML
import html

graph_path = Path(
    "/kaggle/working/ncert_knowledge_graph/knowledge_graph_stable.html"
)

assert graph_path.exists(), "Graph HTML file not found"

graph_html = graph_path.read_text(encoding="utf-8")

print("Graph file size:", round(len(graph_html) / 1024), "KB")

# Embed the complete HTML document inside an iframe
iframe = f"""
<iframe
    srcdoc="{html.escape(graph_html, quote=True)}"
    width="100%"
    height="850"
    style="border:1px solid #ddd; border-radius:8px;"
    sandbox="allow-scripts allow-same-origin"
></iframe>
"""

display(HTML(iframe))

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path
import urllib.request
import numpy as np

OUT = Path("/kaggle/working/ncert_knowledge_graph")
OUT.mkdir(parents=True, exist_ok=True)

# 1. Download a Devanagari-compatible font
font_path = OUT / "NotoSansDevanagari-Regular.ttf"

if not font_path.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/googlefonts/noto-fonts/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf",
        font_path
    )

font_manager.fontManager.addfont(str(font_path))
hindi_font = font_manager.FontProperties(fname=str(font_path))

# 2. Check graph
assert "G" in globals(), "Run the graph creation cell first."

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# 3. Fixed graph layout
pos = nx.spring_layout(
    G,
    seed=42,
    k=2.2,
    iterations=500
)

# 4. Identify cycle-related edges
cycle_edges = set()

for component in nx.strongly_connected_components(G):
    if len(component) > 1:
        cycle_edges.update(G.subgraph(component).edges())

normal_edges = [
    (u, v) for u, v in G.edges()
    if (u, v) not in cycle_edges
]

# 5. Draw graph
fig, ax = plt.subplots(figsize=(26, 20))

colors = {
    "CH04": "#B8DDF4",
    "CH05": "#BFE8D5",
    "CH06": "#F6D8AD",
    "CH07": "#D6C8F0"
}

node_colors = [
    colors.get(G.nodes[n].get("chapter"), "#D5E5F1")
    for n in G.nodes()
]

nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_size=1250,
    node_color=node_colors,
    edgecolors="#607D8B",
    linewidths=1
)

nx.draw_networkx_edges(
    G, pos, ax=ax,
    edgelist=normal_edges,
    edge_color="#A0AEC0",
    width=0.8,
    alpha=0.35,
    arrows=True,
    arrowsize=10,
    node_size=1250
)

nx.draw_networkx_edges(
    G, pos, ax=ax,
    edgelist=list(cycle_edges),
    edge_color="#D9534F",
    style="dashed",
    width=1.6,
    alpha=0.8,
    arrows=True,
    arrowsize=12,
    node_size=1250
)

# 6. English and Hindi labels
for node, (x, y) in pos.items():
    data = G.nodes[node]

    en = str(data.get("name_en", node))
    hi = str(data.get("name_hi", ""))

    ax.annotate(
        en,
        (x, y),
        xytext=(0, 17),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.8,
            pad=1
        )
    )

    ax.annotate(
        hi,
        (x, y),
        xytext=(0, -17),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=9,
        fontproperties=hindi_font,
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.8,
            pad=1
        )
    )

ax.set_title(
    "NCERT Class 6 Mathematics — English-Hindi Prerequisite Knowledge Graph",
    fontsize=20,
    pad=25
)

ax.axis("off")
plt.tight_layout()

# 7. Save
png_path = OUT / "bilingual_knowledge_graph.png"
pdf_path = OUT / "bilingual_knowledge_graph.pdf"

fig.savefig(png_path, dpi=250, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")

# 8. Display directly in Kaggle
plt.show()

print("PNG saved:", png_path)
print("PDF saved:", pdf_path)
print("Cycle-related edges:", len(cycle_edges))
